## **Bank Nifty Predictability Under Regulatory Expiry-Structure Reform**

### **Notebook 06: Proposed Model Estimation**

[![XGBoost](https://img.shields.io/badge/XGBoost-Proposed-red)](https://xgboost.readthedocs.io/)

---

**Scope:** Train and evaluate the proposed forecasting model incorporating combined technical, cyclical expiry distance, and volatility regime probabilities.

---


### **Pipeline Context**

This is a sequential 9-notebook experimental workflow:

1. **01_data_collection.ipynb**
2. **02_data_cleaning_eda.ipynb**
3. **03_feature_engineering.ipynb**
4. **04_volatility_regime_detection.ipynb**
5. **05_baseline_models.ipynb**
6. **06_proposed_model.ipynb**
7. **07_ablation_study.ipynb**
8. **08_evaluation_and_diagnostics.ipynb**
9. **09_figures_for_paper.ipynb**

---
**Current Location:** 6. **06_proposed_model.ipynb** <- *You are here*

---


## 1. Introduction

### 1.1 Purpose of This Notebook
In this notebook, we train the proposed derivatives-structure aware XGBoost model. By adding features representing the cyclical distance to expiry, the weekly expiry flag, and the daily probabilities of the HMM volatility regimes, we evaluate if derivatives cycle mechanics help the model capture next-day price movements more accurately.

## 2. Environment Setup

In [1]:
import sys
from pathlib import Path
import pandas as pd
import joblib

project_dir = Path("e:/BN_Structure_Reforms/banknifty-expiry-volatility-ml")
if str(project_dir) not in sys.path:
    sys.path.append(str(project_dir))

from src import config
from src.models import build_xgb_regressor, train_model
from src.evaluation import calculate_metrics

## 3. Data Preparation

In [2]:
train = pd.read_parquet(config.PROCESSED_DATA_DIR / "train.parquet")
val = pd.read_parquet(config.PROCESSED_DATA_DIR / "val.parquet")
test = pd.read_parquet(config.PROCESSED_DATA_DIR / "test.parquet")

proposed_features = [
    'RSI', 'MACD', 'MACD_Signal', 'MACD_Hist', 'BB_Mid', 'BB_Upper', 'BB_Lower', 'BB_PctB', 'BB_Width', 'ATR', 'Rolling_Vol', 'Return',
    'Days_To_Expiry', 'Expiry_Week', 'Expiry_Dist_Sin', 'Expiry_Dist_Cos',
    'Regime_Prob_Calm', 'Regime_Prob_Stressed'
]

X_train, y_train = train[proposed_features], train['Target']
X_val, y_val = val[proposed_features], val['Target']
X_test, y_test = test[proposed_features], test['Target']

## 4. Proposed Model Training

In [3]:
proposed_model = build_xgb_regressor()
train_model(proposed_model, X_train, y_train, X_val, y_val)
print("[OK] Proposed model trained successfully.")

[OK] Proposed model trained successfully.


## 5. Proposed Model Evaluation

In [4]:
test_preds = proposed_model.predict(X_test)
metrics = calculate_metrics(y_test, test_preds)
print("Proposed model test set metrics:", metrics)
joblib.dump(proposed_model, config.MODELS_DIR / "proposed_xgb.joblib")

Proposed model test set metrics: {'RMSE': 0.012310273811484794, 'MAE': 0.008766360757224242, 'Directional_Accuracy': 0.5306122448979592}


['E:\\BN_Structure_Reforms\\banknifty-expiry-volatility-ml\\models\\proposed_xgb.joblib']